In [113]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [114]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

from tqdm import tqdm

In [115]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print(train.shape)
print(test.shape)

train.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [116]:
OPTIONS = ['A', 'B', 'C', 'D', 'E']

def expand_df(df, is_train=True):

    rows = []

    for _, row in tqdm(df.iterrows(), total=len(df)):

        for option in OPTIONS:

            sample = {
                'id': row['id'],
                'prompt': row['prompt'],
                'option': row[option],
                'option_label': option
            }

            if is_train:
                sample['target'] = int(option == row['answer'])

            rows.append(sample)

    return pd.DataFrame(rows)

In [117]:
train_expanded = expand_df(train, is_train=True)
test_expanded = expand_df(test, is_train=False)

print(train_expanded.shape)
print(test_expanded.shape)

train_expanded.head()

100%|██████████| 500/500 [00:00<00:00, 13789.43it/s]

(10000, 5)
(2500, 4)


,id,prompt,option,option_label,target
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,A,0
1,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans do not e...,B,1
2,1,Pick the best possible answer: What is Martin ...,Martin Heidegger does not believe in the exist...,C,0
3,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that the relationshi...,D,0
4,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that time is an illu...,E,0


In [118]:
train_expanded['text'] = (
    train_expanded['prompt'].astype(str)
    + ' [SEP] '
    + train_expanded['option'].astype(str)
)

test_expanded['text'] = (
    test_expanded['prompt'].astype(str)
    + ' [SEP] '
    + test_expanded['option'].astype(str)
)

In [119]:
def apk(actual, predicted, k=3):

    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0

    for i, p in enumerate(predicted):

        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score


def mapk(actuals, predictions, k=3):

    return np.mean([
        apk(a, p, k)
        for a, p in zip(actuals, predictions)
    ])

In [120]:
vectorizer = TfidfVectorizer(
    max_features=200000,
    ngram_range=(1,3),
    lowercase=True,
    strip_accents="unicode",
    sublinear_tf=True,
    min_df=2,
    max_df=0.95
)



X = vectorizer.fit_transform(train_expanded['text'])

y = train_expanded['target']

print(X.shape)

(10000, 28019)


In [121]:
from sklearn.model_selection import GroupKFold
from sklearn.svm import LinearSVC

N_SPLITS = 5

gkf = GroupKFold(n_splits=N_SPLITS)

groups = train_expanded['id']

oof_probs = np.zeros(len(train_expanded))

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(
    gkf.split(X, y, groups)
):

    print('=' * 50)
    print(f'FOLD {fold + 1}')

    X_train = X[train_idx]
    y_train = y.iloc[train_idx]

    X_val = X[val_idx]
    y_val = y.iloc[val_idx]

    model = LogisticRegression(
        C=5.0,
        max_iter=5000,
        solver="saga",
        class_weight="balanced",
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    val_probs = model.predict_proba(X_val)[:, 1]

    oof_probs[val_idx] = val_probs

    val_df = train_expanded.iloc[val_idx].copy()

    val_df['prob'] = val_probs

    actuals = []
    predictions = []

    for question_id, group in val_df.groupby('id'):

        group = group.sort_values(
            'prob',
            ascending=False
        )

        pred_labels = (
            group['option_label']
            .head(3)
            .tolist()
        )

        actual_label = (
            group[group['target'] == 1]
            ['option_label']
            .iloc[0]
        )

        actuals.append(actual_label)
        predictions.append(pred_labels)

    fold_map3 = mapk(actuals, predictions)

    fold_scores.append(fold_map3)

    print(f'Fold MAP@3: {fold_map3:.5f}')

print('\nFINAL CV MAP@3:', np.mean(fold_scores))

FOLD 1
Fold MAP@3: 0.99500
FOLD 2
Fold MAP@3: 0.99125
FOLD 3
Fold MAP@3: 0.98875
FOLD 4
Fold MAP@3: 0.99083
FOLD 5
Fold MAP@3: 0.98750

FINAL CV MAP@3: 0.9906666666666666


In [123]:
X_train_full = vectorizer.fit_transform(
    train_expanded['text']
)

y_train_full = train_expanded['target']

X_test = vectorizer.transform(
    test_expanded['text']
)

final_model = LogisticRegression(
    C=5,
    max_iter=5000,
    class_weight='balanced',
    n_jobs=-1
)

final_model.fit(X_train_full, y_train_full)

test_probs = final_model.predict_proba(X_test)[:, 1]

test_expanded['prob'] = test_probs

In [125]:
submission_rows = []

for question_id, group in test_expanded.groupby('id'):

    group = group.sort_values(
        'prob',
        ascending=False
    )

    top3 = (
        group['option_label']
        .head(3)
        .tolist()
    )

    prediction = ' '.join(top3)

    submission_rows.append({
        'ID': question_id,
        'Prediction': prediction
    })

submission = pd.DataFrame(submission_rows)

submission.head()

,ID,Prediction
0,1,A E B
1,2,B E D
2,3,B E D
3,4,E C D
4,5,C A D


In [127]:
submission.to_csv(
    'submission.csv',
    index=False
)

print('submission.csv saved')

submission.csv saved
